In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score, log_loss
from catboost import CatBoostClassifier
import mlflow

TABLE_NAME = 'clean_users_churn'

TRACKING_SERVER_HOST = '127.0.0.1'
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = 'churn_laptev_ilya_sergeevich_2'
RUN_NAME_GS = 'model_grid_search'
RUN_NAME_RS = 'model_rnd_search'
REGISTRY_MODEL_NAME_GS = 'cb_grid_search'
REGISTRY_MODEL_NAME_RS = 'cb_rnd_search'



In [2]:
# Загрузка данных
DB_DESTINATION_USER = os.environ["DB_DESTINATION_USER"]
DB_DESTINATION_PASSWORD = os.environ["DB_DESTINATION_PASSWORD"]
DB_DESTINATION_HOST = os.environ["DB_DESTINATION_HOST"]
DB_DESTINATION_PORT = os.environ["DB_DESTINATION_PORT"]
DB_DESTINATION_NAME = os.environ["DB_DESTINATION_NAME"]


conn = create_engine(f"postgresql://{DB_DESTINATION_USER}:{DB_DESTINATION_PASSWORD}@{DB_DESTINATION_HOST}:{DB_DESTINATION_PORT}/{DB_DESTINATION_NAME}")
df = pd.read_sql(f"select * from {TABLE_NAME}", conn)

In [6]:
features = ["monthly_charges", "total_charges", "senior_citizen"]
target = "target"

split_column = 'begin_date'
stratify_column = 'target'
test_size = 0.2

df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(df[features],
                                                    df[target],
                                                    test_size=test_size,
                                                    shuffle=False)

print(f"Размер выборки для обучения: {X_train.shape}")
print(f"Размер выборки для теста: {X_test.shape}")

Размер выборки для обучения: (5615, 3)
Размер выборки для теста: (1404, 3)


In [7]:
loss_function = "Logloss"
task_type = 'CPU'
random_seed = 0
iterations = 300
verbose = False

params = {
    'depth': [None, 3, 4, 5, 8, 10],
    'learning_rate': [0.01, 0.1, 0.3, 0.7],
    'l2_leaf_reg': [1, 3, 5],
    'bagging_temperature': [0.0, 0.5, 1.0]
}

model = CatBoostClassifier(loss_function=loss_function,
                           iterations=iterations,
                           task_type=task_type,
                           random_seed=random_seed,
                           verbose=verbose)

cv = GridSearchCV(estimator=model,
                  param_grid=params, 
                  n_jobs=-1, 
                  cv=2)

clf = cv.fit(X_train, y_train)

In [8]:
cv_results = pd.DataFrame(clf.cv_results_)

best_params = clf.best_params_

model_best = CatBoostClassifier(**best_params, loss_function=loss_function,
                                iterations=iterations, task_type=task_type,
                                verbose=verbose, random_seed=random_seed)

model_best.fit(X_train, y_train)

prediction = model_best.predict(X_test)
probas = model_best.predict_proba(X_test)[:, 1]

In [10]:
# расчёт метрик качества
metrics = {}

_, err1, _, err2 = confusion_matrix(y_test, prediction, normalize='all').ravel()
auc = roc_auc_score(y_test, probas)
precision = precision_score(y_test, prediction)
recall = recall_score(y_test, prediction)
f1 = f1_score(y_test, prediction)
logloss = log_loss(y_test, prediction)

# сохранение метрик в словарь
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

# дополнительные метрики из результатов кросс-валидации
metrics['mean_fit_time'] = cv_results['mean_fit_time'].mean()
metrics['std_fit_time'] = cv_results['std_fit_time'].mean()
metrics['mean_test_score'] = cv_results['mean_test_score'].mean()
metrics['std_test_score'] = cv_results['std_test_score'].mean()
metrics['best_score'] = clf.best_score_

In [13]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = 'https://storage.yandexcloud.net'
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv('AWS_SECRET_ACCESS_KEY')


mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

In [14]:
# настройки для логирования в MLFlow
pip_requirements = '../requirements.txt'
signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME_GS, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    mlflow.log_params(best_params)
    mlflow.log_metrics(metrics)
    
    cv_info = mlflow.sklearn.log_model(cv, artifact_path='cv')
    model_info = mlflow.catboost.log_model(artifact_path='models',
                                           cb_model=model_best,
                                           signature=signature,
                                           input_example=input_example,
                                           registered_model_name=REGISTRY_MODEL_NAME_GS,
                                           await_registration_for=60,
                                           pip_requirements=pip_requirements)
    
    print(f'RUN_ID for {RUN_NAME_GS} is {run_id}')

Successfully registered model 'cb_grid_search'.
2024/11/10 21:52:46 INFO mlflow.tracking._model_registry.client: Waiting up to 60 seconds for model version to finish creation. Model name: cb_grid_search, version 1


RUN_ID for model_grid_search is 35d3946729f04314acb68be52b559e6a


Created version '1' of model 'cb_grid_search'.


In [15]:
# Загрузка данных
DB_DESTINATION_USER = os.environ["DB_DESTINATION_USER"]
DB_DESTINATION_PASSWORD = os.environ["DB_DESTINATION_PASSWORD"]
DB_DESTINATION_HOST = os.environ["DB_DESTINATION_HOST"]
DB_DESTINATION_PORT = os.environ["DB_DESTINATION_PORT"]
DB_DESTINATION_NAME = os.environ["DB_DESTINATION_NAME"]


conn = create_engine(f"postgresql://{DB_DESTINATION_USER}:{DB_DESTINATION_PASSWORD}@{DB_DESTINATION_HOST}:{DB_DESTINATION_PORT}/{DB_DESTINATION_NAME}")
df = pd.read_sql(f"select * from {TABLE_NAME}", conn)

In [16]:
features = ["monthly_charges", "total_charges", "senior_citizen"]
target = "target"

split_column = 'begin_date'
stratify_column = 'target'
test_size = 0.2

df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(df[features],
                                                    df[target],
                                                    test_size=test_size,
                                                    shuffle=False)

print(f"Размер выборки для обучения: {X_train.shape}")
print(f"Размер выборки для теста: {X_test.shape}")

Размер выборки для обучения: (5615, 3)
Размер выборки для теста: (1404, 3)


In [17]:
loss_function = "Logloss"
task_type = 'CPU'
random_seed = 0
iterations = 300
verbose = False

param_distributions = {
    'depth': [None, 3, 4, 5, 8, 10],
    'learning_rate': [0.01, 0.1, 0.3, 0.7],
    'l2_leaf_reg': [1, 3, 5],
    'bagging_temperature': [0.0, 0.5, 1.0]
}

model = CatBoostClassifier(loss_function=loss_function,
                           task_type=task_type,
                           random_seed=random_seed,
                           iterations=iterations,
                           verbose=verbose)

cv = RandomizedSearchCV(estimator=model,
                        param_distributions=param_distributions,
                        n_iter=20,
                        cv=2,
                        n_jobs=-1)

clf = cv.fit(X_train, y_train)

In [18]:
cv_results = pd.DataFrame(clf.cv_results_)

best_params = clf.best_params_

model_best = CatBoostClassifier(**best_params,
                                loss_function=loss_function,
                                task_type=task_type,
                                random_seed=random_seed,
                                iterations=iterations,
                                verbose=verbose)

model_best.fit(X_train, y_train)

prediction = model_best.predict(X_test)
probas = model_best.predict_proba(X_test)[:, 1]

# расчёт метрик качества
metrics = {}

_, err1, _, err2 =  confusion_matrix(y_test, prediction, normalize='all').ravel()
auc = roc_auc_score(y_test, probas)
precision = precision_score(y_test, prediction)
recall = recall_score(y_test, prediction)
f1 = f1_score(y_test, prediction)
logloss = log_loss(y_test, prediction)

# сохранение метрик в словарь
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

# дополнительные метрики из результатов кросс-валидации
metrics["mean_fit_time"] = cv_results['mean_fit_time'].mean()
metrics["std_fit_time"] =  cv_results['std_fit_time'].mean()
metrics["mean_test_score"] = cv_results['mean_test_score'].mean()
metrics["std_test_score"] = cv_results['std_test_score'].mean()
metrics['best_score'] = clf.best_score_

In [19]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = 'https://storage.yandexcloud.net'
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv('AWS_SECRET_ACCESS_KEY')


mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

In [20]:
# настройки для логирования в MLFlow
pip_requirements= '../requirements.txt'
signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME_RS, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    mlflow.log_params(best_params)
    mlflow.log_metrics(metrics)
    
    cv_info = mlflow.sklearn.log_model(cv, artifact_path='cv')
    model_info = mlflow.catboost.log_model(cb_model=model_best,
                                           artifact_path='models',
                                           signature=signature,
                                           input_example=input_example,
                                           registered_model_name=REGISTRY_MODEL_NAME_RS,
                                           await_registration_for=60,
                                           pip_requirements=pip_requirements)
    
    print(f'RUN_ID for {RUN_NAME_GS} is {run_id}')

Successfully registered model 'cb_rnd_search'.
2024/11/10 21:56:02 INFO mlflow.tracking._model_registry.client: Waiting up to 60 seconds for model version to finish creation. Model name: cb_rnd_search, version 1


RUN_ID for model_grid_search is 3b77cd6e3b3e4d20b4bf3929c07fdba7


Created version '1' of model 'cb_rnd_search'.
